In [1]:
import json
from elasticsearch import Elasticsearch
es = Elasticsearch('localhost:9200')

In [2]:
def check_elastic_errors(res):
    error = res['errors']
    if error:
        print(res)
    return error

In [3]:
# use distinct index names, please don't mess into my index :)
indexname="benczur_twitter"

In [13]:
es.indices.delete(index=indexname, ignore=[400, 404])

{'acknowledged': True}

### Read a JSON for Elasticsearch indexing

In [5]:
messages = str('')
with open('/home/benczur/WWWMat2026/Data/climate-tweets1000.json') as f:
    for line in f:
        tweet=json.loads(line.rstrip())
        dict = {
            'tweet_id': tweet['id'],
            'screen_name': tweet['user']['screen_name'],
            'text': tweet['text'],
            'followers_count': tweet['user']['followers_count'],
            'friends_count': tweet['user']['friends_count'],
            'hashtags': ''
        }
        for hashtag in tweet['entities']['hashtags']:
            dict['hashtags']+= hashtag['text']+' '
        messages+=('{ "index":{"_index": "' + indexname + '"}}\n'+\
            json.dumps(dict, ensure_ascii=True) +'\n')
print(messages)

{ "index":{"_index": "benczur_twitter"}}
{"tweet_id": 1099069240538148869, "screen_name": "AussieAce_", "text": "RT @PaulEDawson: A mere 100 companies are responsible for 71 percent of global climate emissions.\n\nThese people are locking you and everyth\u2026", "followers_count": 614, "friends_count": 613, "hashtags": ""}
{ "index":{"_index": "benczur_twitter"}}
{"tweet_id": 1099069238273339393, "screen_name": "kmariagfrunt", "text": "RT @PaulEDawson: \"Greenhouse gases are increasingly disrupting the jet stream, a powerful river of winds that steers weather systems in the\u2026", "followers_count": 892, "friends_count": 1296, "hashtags": ""}
{ "index":{"_index": "benczur_twitter"}}
{"tweet_id": 1099069200868487168, "screen_name": "eyeofthegoddess", "text": "RT @NWPinPDX: #ClimateChange is the \u201choax\u201d that will sink your house.\n\nThe \u201choax\u201d that will burn your neighborhood, ruin your future, take\u2026", "followers_count": 10266, "friends_count": 10109, "hashtags":

In [6]:
res = es.bulk(body=messages)
print(check_elastic_errors(res)) # to check if there is some error

False


### Seach

In [7]:
topic_title = "SimplyTasheena"
queries = [{"query_string": 
              {"query": topic_title}}]

In [8]:
queries = [{"match": {"text": "government"}},
           {"match": {"hashtags" : "energy"}},
           {"match": {"screen_name" : "GCCThinkActTank"}},
           {"match": {"screen_name" : "imagine_garden"}},
           {"range": {"followers_count": {"gte": 100, "lte": 1000}}},
           {"bool": {"must": [{"range": {"followers_count": {"gte": 100, "lte": 1000}}},
                              {"match": {"text": "government"}}]}}
          ]

In [11]:
for index, es_query in enumerate(queries):
    print('Query:', es_query)
    query_response = es.search(index=indexname, 
                               query=es_query)

    print('{} documents found.'.format(query_response['hits']['total']))
    for searchresult in query_response['hits']['hits']:
        print (searchresult['_source'], searchresult['_score'])
    print('\n\n')

Query: {'match': {'text': 'government'}}
{'value': 6, 'relation': 'eq'} documents found.
{'tweet_id': 1099053701308735488, 'screen_name': 'LynnInCA', 'text': 'RT @highcountrynews: Inspired by @GretaThunberg, every Friday @Havenruthie has been striking in front of businesses and government building…', 'followers_count': 203, 'friends_count': 390, 'hashtags': ''} 5.24189
{'tweet_id': 1099068986896105472, 'screen_name': 'GonnaFry', 'text': "Jagmeet Singh joins with Canada's oilygarchs to embrace government subsidies for BC LNG fossil fuels project… https://t.co/P8BtWv7gZO", 'followers_count': 429, 'friends_count': 289, 'hashtags': ''} 5.1321063
{'tweet_id': 1099067714885373952, 'screen_name': 'emilyhamilton86', 'text': 'Massively missed opportunity not to include a call to government to make #greeninfrastructure a national infrastruc… https://t.co/39zBc5MzYz', 'followers_count': 1128, 'friends_count': 949, 'hashtags': 'greeninfrastructure '} 5.1321063
{'tweet_id': 1099065280590958592, 'sc

In [12]:
query_response

{'took': 60,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 4, 'relation': 'eq'},
  'max_score': 6.24189,
  'hits': [{'_index': 'benczur_twitter',
    '_type': '_doc',
    '_id': '9s4pGp0BtMmDfvhc_ja_',
    '_score': 6.24189,
    '_source': {'tweet_id': 1099053701308735488,
     'screen_name': 'LynnInCA',
     'text': 'RT @highcountrynews: Inspired by @GretaThunberg, every Friday @Havenruthie has been striking in front of businesses and government building…',
     'followers_count': 203,
     'friends_count': 390,
     'hashtags': ''}},
   {'_index': 'benczur_twitter',
    '_type': '_doc',
    '_id': 'uM4pGp0BtMmDfvhc_jO2',
    '_score': 6.1321063,
    '_source': {'tweet_id': 1099068986896105472,
     'screen_name': 'GonnaFry',
     'text': "Jagmeet Singh joins with Canada's oilygarchs to embrace government subsidies for BC LNG fossil fuels project… https://t.co/P8BtWv7gZO",
     'followers_count': 429,
     'frie